## 1. Load Data and Initial Inspection

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/02_Parfumo_Perfumes.csv', index_col=0)

In [2]:
df.head(10)
df.shape # 59,325 rows and 13 columns
print(df.info())

<class 'pandas.DataFrame'>
Index: 59325 entries, 455 to nan
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Name           59324 non-null  str    
 1   Brand          59324 non-null  str    
 2   Release_Year   39009 non-null  float64
 3   Concentration  12483 non-null  str    
 4   Rating_Value   30046 non-null  float64
 5   Rating_Count   30046 non-null  float64
 6   Main_Accords   32225 non-null  str    
 7   Top_Notes      31139 non-null  str    
 8   Middle_Notes   31149 non-null  str    
 9   Base_Notes     31154 non-null  str    
 10  Perfumers      20544 non-null  str    
 11  URL            59325 non-null  str    
dtypes: float64(3), str(9)
memory usage: 5.9+ MB
None


Throughout this notebook, the term fragrance is used to refer to all entries in the dataset, regardless of concentration or product type.

**Dataset Summary**  
Data contains 59,325 fragrances released between 1709 and 2024. It represents 1,451 unique brands.  


**Columns**  

| Column | Description |
|--------|-------------|
| Number | Internal Parfumo ID |
| Name | Perfume name |
| Brand | Fragrance house |
| Release_Year | Year first released |
| Concentration | EdT, EdP, Parfum, Cologne, After Shave, etc. |
| Rating_Value | Average user rating |
| Rating_Count | Number of user ratings |
| Main_Accords | Dominant scent families |
| Top / Middle / Base_Notes | Notes smelled over time |
| Perfumers | Creator name(s) |
| URL | Link to the full entry |



**Data Types**  
Only three quantitative variables: Release_Year, Rating_Value, and Rating_Count. Rest are string types.

**Missingsness**  
The dataset is highly sparse, with many optional metadata fields missing.



## 2. Data Quality and Missingness

In [3]:
df.isna().mean() * 100

Name              0.001686
Brand             0.001686
Release_Year     34.245259
Concentration    78.958281
Rating_Value     49.353561
Rating_Count     49.353561
Main_Accords     45.680573
Top_Notes        47.511167
Middle_Notes     47.494311
Base_Notes       47.485883
Perfumers        65.370417
URL               0.000000
dtype: float64

### Missing Data
- The dataset is highly sparse, with most missing values in descriptive fields such as notes, accords, perfumers, and concentration. 
- Core identifiers (Name, Brand, URL) are nearly complete.
- Missingness varies strongly across columns: Release_Year has ~34% missing values, while Concentration is missing in ~78% of cases. Rating data is available for roughly half of the entries.
- This pattern suggests systematic differences in data coverage rather than random missingness. 

➡ The full dataset is retained as a baseline, with column-specific subsets used where needed.


## 3. Structural Cleaning 

In [4]:
# Drop rows with missing values in critical columns
df = df.dropna(subset=["Name", "Brand", "URL"], how="any")
df = df[df["Name"] != "NAME?"]

A small number of rows contain no usable identifiers (e.g. missing Name and Brand) or inconsistent structure (broken URLs). These rows are considered invalid records rather than missing data.

These entries are removed from the dataset as they cannot be reliably interpreted or linked to a fragrance entity.

In [5]:
df.duplicated().sum()  # 9 rows are duplicated  completely
df[df.duplicated()]    # all 9 rows are empty placeholder data and can be dropped
df = df.drop_duplicates()

In [6]:
df["URL"].duplicated().sum() # URL are not unique, there are 35 duplicates in the dataset

df[df["URL"].duplicated(keep=False)].sort_values("URL")

# drop all rows with duplicated URL values
df = df[~df["URL"].duplicated(keep=False)]
df["URL"].is_unique 

True

### Handling URL Duplicates (Edge Case: Frau Tonis Parfum)
There are 35 rows sharing duplicate URLs, restricted to a single brand. Manual inspection reveals that these duplicates stem from scraping/extraction errors, combining correct metadata with completely fabricated or misaligned rows (e.g., wrong scent notes or release years). 

Because automated deduplication (like `keep='first'`) would inject corrupted data into the analysis, and manual step-by-step verification is inefficient for this minor subset, all affected rows are strictly removed using `keep=False` to preserve data integrity.

In [7]:
# add data completeness column to the dataframe
df["completeness"] = (df.notna().mean(axis=1) * 100).round(1)

### Thoughts on dropping "Number" column
The Number column is an internal Parfumo identifier.  
Although it is frequently missing, it is retained as a supplementary reference field where available.  
It is not used as a primary key due to incomplete coverage.

URL values are not consistent identifiers in the original dataset due to extraction issues.  
To ensure internal consistency, all ambiguous URL entries were removed, resulting in a subset with unique URL values.  

## 4. Datatypes Conversion and Text Standardization

In [8]:
df["Release_Year"] = df["Release_Year"].astype("Int64")
df["Rating_Count"] = df["Rating_Count"].astype("Int64")
df.info()

<class 'pandas.DataFrame'>
Index: 59239 entries, 455 to nan
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Name           59239 non-null  str    
 1   Brand          59239 non-null  str    
 2   Release_Year   38971 non-null  Int64  
 3   Concentration  12460 non-null  str    
 4   Rating_Value   30001 non-null  float64
 5   Rating_Count   30001 non-null  Int64  
 6   Main_Accords   32173 non-null  str    
 7   Top_Notes      31102 non-null  str    
 8   Middle_Notes   31112 non-null  str    
 9   Base_Notes     31117 non-null  str    
 10  Perfumers      20529 non-null  str    
 11  URL            59239 non-null  str    
 12  completeness   59239 non-null  float64
dtypes: Int64(2), float64(2), str(9)
memory usage: 6.4+ MB


In [38]:
df.head(10)
# df['Perfumers'].unique() 
mask = (df['Main_Accords'].notna()) & (df['Top_Notes'].notna()) & (df['Middle_Notes'].notna()) & (df['Base_Notes'].notna())
df[mask]

df.duplicated(subset=['Name', 'Brand', 'Release_Year']).any()
df.duplicated(subset=['Name', 'Brand', 'Release_Year']).sum()
# df[df.duplicated(subset=['Number', 'Name', 'Brand', 'Release_Year'], keep=False)].sort_values(['Brand', 'Name'])

dupes = df[df.duplicated(subset=['Name', 'Brand', 'Release_Year'], keep=False)].sort_values(['Brand', 'Name'])
pd.set_option('display.max_colwidth', None)
dupes[['Name', 'Brand', 'Release_Year', 'Main_Accords', 'Top_Notes', 'URL']]

df[['Name', 'Brand', 'Release_Year', 'URL']].sample(n=25)


,Name,Brand,Release_Year,URL
Number,,,,
NaN,Big Dreams,bodycology,<NA>,https://www.parfumo.com/Perfumes/bodycology/big-dreams
NaN,Forest,Rook Perfumes,2018,https://www.parfumo.com/Perfumes/Rook_Perfumes/Forest
NaN,Flowerbomb La Vie en Rose 2011,Viktor & Rolf,2011,https://www.parfumo.com/Perfumes/Viktor_Rolf/Flowerbomb_La_Vie_en_Rose_2011
NaN,L'Orchidée,Léonard,1903,https://www.parfumo.com/Perfumes/Leonard/L_Orchidee
NaN,Vetiver & Black Tea,Kiehl's,2014,https://www.parfumo.com/Perfumes/Kiehls/Vetiver__Black_Tea
NaN,Lapsang Su Chong No. 8,Tokyomilk,<NA>,https://www.parfumo.com/Perfumes/Tokyomilk/Lapsang_Su_Chong_No_8
NaN,Make Perfume Not War,Histoires de Parfums,2013,https://www.parfumo.com/Perfumes/Histoires_de_Parfums/Make_Perfume_Not_War
NaN,Carnation Oeillet,Crabtree & Evelyn,<NA>,https://www.parfumo.com/Perfumes/Crabtree__Evelyn/Carnation_Oeillet
NaN,Salvatore Ferragamo pour Homme Salvatore Ferragamo 1999 Eau de Toilette,Salvatore Ferragamo,1999,https://www.parfumo.com/Perfumes/Salvatore_Ferragamo/Salvatore_Ferragamo_pour_Homme_Eau_de_Toilette


Strings bereinigen (.str.strip(), Lowercase-Check für Kategorien/Notes).

String-Cleaning (Sehr wichtig hier): Du hast Spalten wie Top_Notes ("Basil, Bergamot, Lemon..."). Das sind kommagetrennte Strings. 
Für eine spätere Analyse musst du Whitespaces entfernen. Zudem solltest du prüfen, ob es gemischte Schreibweisen gibt (z.B. "jasmine" vs. "Jasmine").

Ausreißer-Kontrolle (Outlier): Du erwähnst ein Release_Year von 1709. Ist das ein valider Duft (z.B. das originale Kölnisch Wasser) oder ein Tippfehler? 
Solche Extremwerte müssen im Cleaning validiert werden.

Inkonsistente Text-Missings: Du filterst df["Name"] != "NAME?". Was ist mit leeren Strings (""), Leerzeichen (" ") oder "None" als Text? 
Nutze .strip() und ersetze Scheindaten durch echtes np.nan.


## 5. Exploratory summary

In [10]:
df.shape # 59,239 rows and 14 columns
print(df.describe(include='all') )

          Name  Brand  Release_Year    Concentration  Rating_Value  \
count    59239  59239       38971.0            12460  30001.000000   
unique   55077   1451          <NA>              412           NaN   
top     Chypre   Avon          <NA>  Eau de Toilette           NaN   
freq        37   1000          <NA>             2661           NaN   
mean       NaN    NaN   2006.267173              NaN      7.347768   
std        NaN    NaN      22.86725              NaN      0.933054   
min        NaN    NaN        1709.0              NaN      0.000000   
25%        NaN    NaN        2005.0              NaN      6.900000   
50%        NaN    NaN        2013.0              NaN      7.400000   
75%        NaN    NaN        2018.0              NaN      7.900000   
max        NaN    NaN        2024.0              NaN     10.000000   

        Rating_Count Main_Accords Top_Notes Middle_Notes Base_Notes  \
count        30001.0        32173     31102        31112      31117   
unique          <

- URL links are now unique
- A new completeness column shows how complete a row is in %.
- 86 rows were removed due to duplicates or invalid records.
- Dataset still retains high missing values, I will use subsets where needed.

**Key Statistics**  
Median rating: 7.4  
Median rating count: 19  
Max rating count: 2,732  

➡ Ratings are skewed toward generally positive values.  
➡ Engagement varies strongly across fragrances, with a small subset receiving most ratings. 

## 6. Export Cleaned Data


In [11]:
df.to_csv("data/cleaned_data.csv", index=False)